In [6]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import DataLoader
from dataset import IMDBDataset
from model import AgeClassifier
from torch.utils.data import random_split, SubsetRandomSampler
from torch import optim, nn
from tqdm import tqdm

In [7]:
import tarfile
import os

if not os.path.isdir('./imdb_crop'):
    with tarfile.open('./imdb_crop.tar', 'r') as tar:
        tar.extractall('./')

In [ ]:
dataset = IMDBDataset('./imdb_crop/imdb.mat', './imdb_crop', limit=4096)

indices = list(range(len(dataset)))
split = int(np.floor(0.2 * len(dataset)))
np.random.shuffle(indices)
train_indices, val_indices = indices[split:], indices[:split]

train_sampler = SubsetRandomSampler(train_indices)
val_sampler = SubsetRandomSampler(val_indices)

train_loader = DataLoader(dataset, batch_size=32, sampler=train_sampler)
val_loader = DataLoader(dataset, batch_size=32, sampler=val_sampler)

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AgeClassifier(num_classes=5, pretrained=True).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
num_epochs = 8
for epoch in range(num_epochs):
    # Training phase
    model.train()
    train_loss = 0.0
    train_correct = 0
    
    for images, ages in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} - Train"):
        images, ages = images.to(device), ages.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, ages)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        train_correct += (predicted == ages).sum().item()
    
    # Validation phase
    model.eval()
    val_loss = 0.0
    val_correct = 0
    
    with torch.no_grad():
        for images, ages in val_loader:
            images, ages = images.to(device), ages.to(device)
            outputs = model(images)
            loss = criterion(outputs, ages)
            
            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            val_correct += (predicted == ages).sum().item()
    
    train_acc = 100 * train_correct / len(train_indices)
    val_acc = 100 * val_correct / len(val_indices)
    
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss/len(train_loader):.4f}, Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_loss/len(val_loader):.4f}, Val Acc: {val_acc:.2f}%")
    print("-" * 50)

Epoch 1/4 - Train: 100%|██████████| 52/52 [00:18<00:00,  2.78it/s]


Epoch 1/4
Train Loss: 1.3522, Train Acc: 38.01%
Val Loss: 1.1514, Val Acc: 40.83%
--------------------------------------------------


Epoch 2/4 - Train: 100%|██████████| 52/52 [00:12<00:00,  4.30it/s]


Epoch 2/4
Train Loss: 1.1247, Train Acc: 45.52%
Val Loss: 1.2687, Val Acc: 43.77%
--------------------------------------------------


Epoch 3/4 - Train: 100%|██████████| 52/52 [00:12<00:00,  4.28it/s]


Epoch 3/4
Train Loss: 1.0324, Train Acc: 51.43%
Val Loss: 1.1092, Val Acc: 47.43%
--------------------------------------------------


Epoch 4/4 - Train: 100%|██████████| 52/52 [00:12<00:00,  4.27it/s]


Epoch 4/4
Train Loss: 1.0126, Train Acc: 54.06%
Val Loss: 1.1670, Val Acc: 37.41%
--------------------------------------------------


In [ ]:
checkpoint = {
    'epoch': num_epochs,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'train_loss': train_loss/len(train_loader),
    'val_loss': val_loss/len(val_loader),
    'train_acc': train_acc,
    'val_acc': val_acc,
}

torch.save(checkpoint, 'model_checkpoint.pth')
print(f"Checkpoint saved to model_checkpoint.pth")